# Kisara — uji suara F5 Indonesia di Colab

Notebook ini memakai checkpoint **F5 Indonesia V2**, bukan model dasar Inggris/Mandarin. Semua langkah dijalankan lewat notebook; tidak membuat server atau tautan publik.

**Batas lisensi:** bobot model CC-BY-NC-4.0 (nonkomersial). Untuk konten monetisasi, pastikan memperoleh izin atau memakai checkpoint lain yang sesuai. Mengganti tempat menjalankan model tidak mengubah lisensinya.

1. Pilih **Runtime / Waktu proses → Change runtime type / Ubah jenis runtime → T4 GPU** (atau GPU yang tersedia), lalu Simpan.
2. Jalankan sel kode dari atas ke bawah dengan tombol ▶ di kiri. Tunggu sampai satu sel selesai sebelum lanjut.
3. GPU gratis tidak dijamin tersedia. Jika tidak mendapat GPU, berhenti dan coba lagi nanti.
4. Uji satu kalimat dahulu. Kemiripan suara dan kualitas belum dijamin.

Sampel diunggah ke runtime Google Colab milik sesi kamu. Jangan bagikan notebook beserta output suara pribadi. Setelah selesai, unduh hasil lalu hapus runtime bila tidak diperlukan.

Sumber: [model dan lisensi](https://huggingface.co/Eempostor/F5-TTS-INDO-FINETUNE-V2), [kode F5 resmi](https://github.com/SWivid/F5-TTS), [batas Colab](https://research.google.com/colaboratory/faq.html).


## 1. Periksa GPU dan pasang paket
Unduhan pertama bisa memakan beberapa menit. Kalau diminta restart sesi setelah instalasi, lakukan lalu jalankan kembali mulai sel 2.


In [ ]:
import subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'f5-tts==1.1.22'], check=True)
print('Instalasi selesai. Lanjut sel 2.')


## 2. Muat model Indonesia
Checkpoint sekitar 1,35 GB; tunggu sampai muncul “Model Indonesia siap”. Tidak perlu token Hugging Face untuk checkpoint publik ini.


In [ ]:
import torch
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

if not torch.cuda.is_available():
    raise RuntimeError('GPU belum aktif. Pilih GPU lewat menu Runtime, lalu sambungkan ulang.')
repo = 'Eempostor/F5-TTS-INDO-FINETUNE-V2'
revision = 'ec3e116a9373f83cfc2d53170a72643f596ea6fc'
checkpoint = hf_hub_download(repo, 'f5_tts_indo_v2.pt', revision=revision)
vocabulary = hf_hub_download(repo, 'vocab.txt', revision=revision)
tts = F5TTS(model='F5TTS_Base', ckpt_file=checkpoint, vocab_file=vocabulary, device='cuda')
print('Model Indonesia siap di', torch.cuda.get_device_name(0))


## 3. Pilih sampel suara sendiri
Upload satu MP3/WAV/OGG bersih sepanjang 3–15 detik, tanpa musik, dengan satu orang berbicara. Kamu bisa memakai file yang tadi dipilih di Kisara; tidak perlu merekam ulang bila sudah jelas.


In [ ]:
from google.colab import files
from pathlib import Path
import soundfile as sf
import subprocess

reference_audio = None
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Pilih tepat satu file sampel suara, lalu jalankan ulang sel ini.')
source = Path(next(iter(uploaded))).resolve()
if source.stat().st_size > 20 * 1024 * 1024:
    raise ValueError('Sampel maksimal 20 MB.')
reference_audio = '/content/kisara-reference-normalized.wav'
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', str(source), '-t', '16', '-vn', '-ac', '1', '-ar', '24000', '-c:a', 'pcm_s16le', reference_audio], check=True)
info = sf.info(reference_audio)
if not 3 <= info.duration <= 15:
    reference_audio = None
    raise ValueError('Durasi sampel harus 3–15 detik. Sesuaikan sampel dan transkripnya.')
print(f'Sampel siap: {info.duration:.2f} detik. Lanjut isi transkrip di sel 4.')


## 4. Isi transkrip, kalimat uji dan tempo
Ganti isi kutipan transkrip dengan ucapan **persis** dalam sampel. Jangan biarkan kalimat contoh jika tidak sama. Tempo awal 0.85 lebih pelan dari 1.0, tetapi tetap perlu kamu dengarkan hasilnya.


In [ ]:
#@title Teks dan tempo
reference_text = "" #@param {type:"string"}
narration_text = "Malam itu, aku mendengar tiga ketukan pelan dari balik pintu. Padahal, aku sedang sendirian di rumah." #@param {type:"string"}
speed = 0.85 #@param {type:"number"}

if not reference_text.strip():
    raise ValueError('Isi reference_text dengan ucapan persis dari sampel suara.')
if not narration_text.strip() or len(narration_text) > 400:
    raise ValueError('Untuk uji pertama, isi narasi 1–3 kalimat, maksimal 400 karakter.')
if not 0.6 <= speed <= 1.2:
    raise ValueError('Untuk uji ini gunakan tempo antara 0.6 dan 1.2.')
print('Teks siap. Lanjut sel 5.')


## 5. Buat dan dengarkan suara
Jangan lanjut membuat naskah panjang sebelum pelafalan, kemiripan suara dan temponya cocok. Jalankan ulang sel 4 lalu 5 untuk mengganti teks atau tempo.


In [ ]:
from IPython.display import Audio, display
from pathlib import Path

if not reference_audio or not Path(reference_audio).is_file():
    raise ValueError('Jalankan sel 3 dan pilih sampel yang valid dahulu.')
if not reference_text.strip() or not narration_text.strip() or len(narration_text) > 400:
    raise ValueError('Periksa teks di sel 4 dahulu.')
if not 0.6 <= speed <= 1.2:
    raise ValueError('Periksa tempo di sel 4 dahulu.')
output_path = '/content/kisara-uji-f5-indonesia.wav'
Path(output_path).unlink(missing_ok=True)
waveform, sample_rate, _ = tts.infer(ref_file=reference_audio, ref_text=reference_text.strip(), gen_text=narration_text.strip(), speed=float(speed), nfe_step=32, seed=42, remove_silence=False)
sf.write(output_path, waveform, sample_rate)
print('Uji selesai. Dengarkan dahulu; ini bukan bukti narasi panjang akan selalu berhasil.')
display(Audio(filename=output_path))


## 6. Unduh hasil uji
Ini menghasilkan WAV untuk didengarkan atau dipakai di editor yang mendukung impor audio. Impor narasi Colab dan pemotongan per adegan otomatis **belum tersedia di Kisara**; notebook ini belum terhubung ke aplikasi.


In [ ]:
from google.colab import files
files.download(output_path)
